In [50]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import csv

import time

In [36]:
#headless mode
options = Options()
options.add_argument("--headless")

# 크롬 드라이버 경로 설정
service = Service(executable_path='../chromedriver-mac-arm64/chromedriver')
driver = webdriver.Chrome(service=service, options=options)


# 웹 페이지 열기
driver.get("https://www.chosun.com/economy/real_estate/?page=1")

wait = WebDriverWait(driver, 5)

anchor = wait.until(EC.presence_of_element_located((
    By.CSS_SELECTOR,
    'div.story-card-component a.story-card__headline'
)))


In [37]:

# 최대 몇 번까지 클릭할지 지정
MAX_PAGE = 10  # 예: 1~50까지 탐색
PAGES_PER_SCREEN = 10

current = 1
hrefs = []


while current <= MAX_PAGE:
    for i in range(current, min(current + PAGES_PER_SCREEN, MAX_PAGE + 1)):
        try:
            # 현재 페이지 범위 내에서 버튼 클릭
            page_button = driver.find_element(By.ID, str(i))
            page_button.click()
            print(f"Clicked page {i}")
            time.sleep(3)
        except NoSuchElementException:
            print(f"[INFO] Page button {i} not found (possibly not visible yet)")
        except Exception as e:
            print(f"Failed to click page {i}: {e}")

    links = driver.find_elements(By.CSS_SELECTOR, "a")
    

    for link in links:
        href = link.get_attribute("href")
        if href:
            hrefs.append(href)


    article_links = [href for href in hrefs if "/economy/real_estate/20" in href]
    
    # 다음 페이지 버튼 클릭 (페이지가 끝나지 않았다면)
    if current + PAGES_PER_SCREEN <= MAX_PAGE:
        try:
            next_page_btn = driver.find_element(
                By.XPATH,
                '//*[@id="main"]/div[2]/section/div/div/div/div[21]/div/div[3]/button'
            )
            next_page_btn.click()
            print("Clicked next page button")
            time.sleep(3)
        except Exception as e:
            print(f"Failed to click next page button: {e}")

    # 다음 10개로 이동
    current += PAGES_PER_SCREEN



Clicked page 1
Clicked page 2
Clicked page 3
Clicked page 4
Clicked page 5
Clicked page 6
Clicked page 7
Clicked page 8
Clicked page 9
Clicked page 10


In [38]:
driver.quit()

In [39]:
# 중복되는 기사들의 링크를 수집함을 확인
# set 자료형을 사용해서 중복을 제거함
article_links = set(article_links)
article_links

{'https://www.chosun.com/economy/real_estate/2024/05/09/RSZXYNQZ6JBIJMJYYDWSX2KHQE/',
 'https://www.chosun.com/economy/real_estate/2024/09/13/UCSK4OOGXVC2LA3PN22T7GJ4IA/',
 'https://www.chosun.com/economy/real_estate/2024/10/07/H2TJ332M5FEWLA72MJS4SAJYMI/',
 'https://www.chosun.com/economy/real_estate/2024/11/27/SQ4ZUUHGWFHDTKPVTCPM7G4B3U/',
 'https://www.chosun.com/economy/real_estate/2024/12/18/4KOOH626JZCYNM6WNRXRCFRDNE/',
 'https://www.chosun.com/economy/real_estate/2024/12/19/6SSZ4NUVOJF3TCIP2QPAQAXRQE/',
 'https://www.chosun.com/economy/real_estate/2025/03/04/XZ2KSUJ5SFFAHC6GRSCMSVSQKQ/',
 'https://www.chosun.com/economy/real_estate/2025/03/15/5N7W7XOV3JHIXKSV5OGJT2DDEY/',
 'https://www.chosun.com/economy/real_estate/2025/03/17/KF5WPVFEU5HSPD5676OWH3PFKU/',
 'https://www.chosun.com/economy/real_estate/2025/03/21/ENVE2A5A7FE5NAZZSYO45TS3Y4/',
 'https://www.chosun.com/economy/real_estate/2025/03/21/F6SQW5LJJZCQFJLTK6W473ZSGU/',
 'https://www.chosun.com/economy/real_estate/2025/03/2

In [40]:
len(article_links)

30

In [41]:
article_links = list(article_links)

In [42]:
article_links[2]

'https://www.chosun.com/economy/real_estate/2025/03/24/VUNCVA65B5CHPGZRZIQE5OGWFI/'

In [43]:
# import pickle

# with open('article_links', 'wb') as f:
#     pickle.dump(article_links, f)

In [48]:
# Headless 모드 설정
options = Options()
options.add_argument("--headless")

# 크롬 드라이버 경로 설정
service = Service(executable_path='../chromedriver-mac-arm64/chromedriver')
driver = webdriver.Chrome(service=service, options=options)

# 기사 본문 수집하는 딕셔너리
article = {}

for i in range(len(article_links)):
    # 웹 페이지 열기
    driver.get(article_links[i])
    time.sleep(2)
    try:
        # 기사 본문이 있는 section 선택
        article_section = driver.find_element(By.CSS_SELECTOR, 'section.article-body')
        
        # 모든 <p> 태그에서 텍스트 누적
        full_text = ""
        paragraphs = article_section.find_elements(By.TAG_NAME, 'p')
        
        for p in paragraphs:
            text = p.text.strip()
            if text:
                full_text += text
    finally:
        # 딕셔너리에 URL을 키로, 본문을 값으로 저장
        article[article_links[i]] = full_text
        
driver.quit()

In [49]:
article

{'https://www.chosun.com/economy/real_estate/2025/03/24/RTHKCAVKV5A5BOIEHHMH2XDTW4/': '국토교통부가 최근 6개월(작년 9월~2월)간 공동주택 하자 건수 상위 20개 건설사 명단을 공개했다. 전체 1위는 97건을 기록한 한화 건설부문이었다. 10대 건설사 중에선 현대건설(전체 2위), 대우건설(전체 14위) 두 곳이 포함됐다.국토부는 2023년 하반기부터 하심위 판정을 근거로 6개월마다 하자가 많은 건설사 리스트를 발표하고 있다. 지난해 하반기 발표에선 현대건설과 대우건설을 포함해 10대 건설사가 네 곳 있었는데, 이번에 현대엔지니어링과 포스코이앤씨가 빠졌다.24일 국토부에 따르면, 하자심사분쟁조정위원회(하심위)가 지난해 9월부터 올 2월까지 6개월간 공동주택 하자 접수 건을 심사한 결과 시공 능력 평가 11위인 한화 건설부문이 판정 건수 97건으로 가장 많은 것으로 나타났다.33개 분쟁 사건의 하자 195건에 대해 심사가 이뤄졌고, 최종 16개 분쟁 건의 97건이 하자로 판정됐다. 가구 수 대비 하자 판정 비율은 8.9%(1091가구 중 97건)였다. 한화 관계자는 “한 아파트 단지 주민들이 놀이터 시설과 관련해 단체로 하자 신청을 제기했고, 현재는 보수를 완료한 상황”이라고 설명했다.10대 건설사 중에선 시평 2위 현대건설이 81건으로 하자 건수 2위를 기록했다. 지난 평가에서 36건으로 18위였는데, 이번에 2배 이상으로 늘어났다. 현대건설 관계자는 “10여 가구가 마루, 벽지찍힘 등을 하자로 신고한 것으로 파악했고, 설계나 시공 관련 하자는 없었다”고 말했다.대우건설은 20건으로 14위였다. 지난 평가에선 51건으로 10위였는데, 하자 판정 건수가 절반 이상 감소했다.하자 건수 전체 3위는 대우조선해양건설(80건)이었고, 한경기건(79건), 삼부토건(71건)이 4위, 5위를 했다.공급 가구 수 대비 하자 판정 비율 순위에선 10대 건설사가 없었다. 판정 건수 1위인 한화 건설 부문이 판정

In [51]:
# 크롤링 내용 csv로 저장

with open('../data/raw/news/chosun_ilbo.csv', 'w', newline="", encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['URL', 'content'])

    for url, content in article.items():
        writer.writerow([url, content])
    

In [53]:
import pandas as pd
chosun = pd.read_csv("../data/raw/news/chosun_ilbo.csv")

In [57]:
chosun['content'].head(1)

0    국토교통부가 최근 6개월(작년 9월~2월)간 공동주택 하자 건수 상위 20개 건설사...
Name: content, dtype: object

In [60]:
# 긴 텍스트 전부 출력하려면 아래 옵션 설정
pd.set_option('display.max_colwidth', None)

# 예시: 긴 텍스트가 들어있는 시리즈 출력
print(chosun['content'].head(1))

0    국토교통부가 최근 6개월(작년 9월~2월)간 공동주택 하자 건수 상위 20개 건설사 명단을 공개했다. 전체 1위는 97건을 기록한 한화 건설부문이었다. 10대 건설사 중에선 현대건설(전체 2위), 대우건설(전체 14위) 두 곳이 포함됐다.국토부는 2023년 하반기부터 하심위 판정을 근거로 6개월마다 하자가 많은 건설사 리스트를 발표하고 있다. 지난해 하반기 발표에선 현대건설과 대우건설을 포함해 10대 건설사가 네 곳 있었는데, 이번에 현대엔지니어링과 포스코이앤씨가 빠졌다.24일 국토부에 따르면, 하자심사분쟁조정위원회(하심위)가 지난해 9월부터 올 2월까지 6개월간 공동주택 하자 접수 건을 심사한 결과 시공 능력 평가 11위인 한화 건설부문이 판정 건수 97건으로 가장 많은 것으로 나타났다.33개 분쟁 사건의 하자 195건에 대해 심사가 이뤄졌고, 최종 16개 분쟁 건의 97건이 하자로 판정됐다. 가구 수 대비 하자 판정 비율은 8.9%(1091가구 중 97건)였다. 한화 관계자는 “한 아파트 단지 주민들이 놀이터 시설과 관련해 단체로 하자 신청을 제기했고, 현재는 보수를 완료한 상황”이라고 설명했다.10대 건설사 중에선 시평 2위 현대건설이 81건으로 하자 건수 2위를 기록했다. 지난 평가에서 36건으로 18위였는데, 이번에 2배 이상으로 늘어났다. 현대건설 관계자는 “10여 가구가 마루, 벽지찍힘 등을 하자로 신고한 것으로 파악했고, 설계나 시공 관련 하자는 없었다”고 말했다.대우건설은 20건으로 14위였다. 지난 평가에선 51건으로 10위였는데, 하자 판정 건수가 절반 이상 감소했다.하자 건수 전체 3위는 대우조선해양건설(80건)이었고, 한경기건(79건), 삼부토건(71건)이 4위, 5위를 했다.공급 가구 수 대비 하자 판정 비율 순위에선 10대 건설사가 없었다. 판정 건수 1위인 한화 건설 부문이 판정 비율 순위에선 8.9%로 11위를 했다.판정 비율 순위 1위는 삼도종합건설로 8가구에서 64건이 나와 가구 수 대비 판정 비율 800%를 기록했다